# ALS: влияет ли поправка за оценку на качество?

Меняем **только score_coef: 0 / 0,2 / 0,4**. Сравнение на одном warm-validation,
с тем же train, фильтром 5/5, весами статусов и конфигурацией ALS из предыдущего
опыта. Для каждого коэффициента используем одинаковые seed 42, 7, 2026.

При score_coef=0 оценка ниже 5 всё ещё обнуляет вес; отключается только
относительная поправка. При коэффициентах 0,2 и 0,4 личное/общее среднее
вычисляется **только по train**. Затухание выключено.

Проверяем mean NDCG@10 по трём seed, разброс и парные разницы с исходным 0,2.
Это оценка чувствительности к инициализации, не независимые тестовые выборки.
Test не оценивается, прежние результаты и модель не перезаписываются.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
import json
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.6f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
COEFFICIENTS = [0.0, 0.2, 0.4]
SEEDS = [42, 7, 2026]
ALS_PARAMS = dict(factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4)
versions = {name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]}
print(versions)
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
users = sorted(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print(f"Train после фильтра: {len(training):,} записей; warm-validation: {len(users):,} пользователей")

/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


Train после фильтра: 3,499,908 записей; warm-validation: 7,184 пользователей


## Контрольный baseline

В этой серии меняется только ALS. Популярность за 28 дней должна воспроизводить
предыдущий результат ≈ 0,113 NDCG@10.

In [2]:
popular = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
popular_metrics = evaluate(popular.recommend_many(users, k=10), targets)
pd.Series(popular_metrics, name="популярность 28 дней").to_frame()

,популярность 28 дней
precision@10,0.099624
recall@10,0.064522
ndcg@10,0.113021
coverage@10,65.000000
users,"7,184.000000"


## Девять запусков

Для всех коэффициентов проверяем, что набор положительных пар одинаковый —
меняются веса, а не состав обучения. История исключений не меняется.
Пустая выдача неизвестному пользователю остаётся нулём; fallback не добавляем,
чтобы измерить эффект на самой ALS.

In [3]:
records = []
per_user = {}
reference_pairs = None
for coef in COEFFICIENTS:
    weighted = weighted_interactions(training, score_coef=coef)
    pairs = weighted[["user_id", "anime_id"]]
    if reference_pairs is None:
        reference_pairs = pairs.copy()
    else:
        pd.testing.assert_frame_equal(pairs, reference_pairs)
    for seed in SEEDS:
        started = perf_counter()
        model = ALSRecommender(**ALS_PARAMS, random_state=seed).fit(weighted, history=history)
        fit_seconds = perf_counter() - started
        started = perf_counter()
        recs = model.recommend_many(users, k=10)
        recommend_seconds = perf_counter() - started
        assert all(len(r) == len(set(r)) and not (set(r) & seen.get(u, set())) for u, r in recs.items())
        metrics = evaluate(recs, targets)
        per_user[(coef, seed)] = np.array([ndcg_at_k(recs[u], targets[u], 10) for u in users])
        records.append({"score_coef": coef, "seed": seed, **metrics,
                        "training_rows": len(weighted),
                        "short_lists": sum(len(r) < 10 for r in recs.values()),
                        "fit_seconds": fit_seconds, "recommend_seconds": recommend_seconds})
        print(f"coef={coef}, seed={seed}: NDCG@10={metrics['ndcg@10']:.6f}, "
              f"coverage={metrics['coverage@10']}, train={fit_seconds:.1f}s", flush=True)
results = pd.DataFrame(records)
display(results.set_index(["score_coef", "seed"]))

coef=0.0, seed=42: NDCG@10=0.100216, coverage=4253, train=7.3s


coef=0.0, seed=7: NDCG@10=0.099340, coverage=4219, train=7.1s


coef=0.0, seed=2026: NDCG@10=0.099730, coverage=4256, train=7.3s


coef=0.2, seed=42: NDCG@10=0.099800, coverage=4199, train=6.9s


coef=0.2, seed=7: NDCG@10=0.098891, coverage=4192, train=7.1s


coef=0.2, seed=2026: NDCG@10=0.099535, coverage=4226, train=7.1s


coef=0.4, seed=42: NDCG@10=0.099532, coverage=4188, train=6.9s


coef=0.4, seed=7: NDCG@10=0.098389, coverage=4162, train=6.7s


coef=0.4, seed=2026: NDCG@10=0.099168, coverage=4214, train=7.3s


precision@10  recall@10  ndcg@10  coverage@10  users  \
score_coef seed                                                         
0.000000   42        0.081974   0.064405 0.100216         4253   7184   
           7         0.081584   0.063838 0.099340         4219   7184   
           2026      0.081807   0.064172 0.099730         4256   7184   
0.200000   42        0.081779   0.065096 0.099800         4199   7184   
           7         0.081361   0.063266 0.098891         4192   7184   
           2026      0.081514   0.063555 0.099535         4226   7184   
0.400000   42        0.081668   0.065438 0.099532         4188   7184   
           7         0.081347   0.063163 0.098389         4162   7184   
           2026      0.081111   0.063276 0.099168         4214   7184   

                 training_rows  short_lists  fit_seconds  recommend_seconds  
score_coef seed                                                              
0.000000   42          3293589            0     7.339438           9.197319  
           7           3293589            0     7.088982           9.128272  
           2026        3293589            0     7.335544           9.144458  
0.200000   42          3293589            0     6.913836           8.937496  
           7           3293589            0     7.093291           9.040634  
           2026        3293589            0     7.105414           9.017686  
0.400000   42          3293589            0     6.878987           8.889075  
           7           3293589            0     6.689963           9.146138  
           2026        3293589            0     7.272979           8.941926

## Среднее качество и устойчивость

Std описывает разброс между тремя инициализациями на одном validation.
Выбираем предварительного кандидата по среднему NDCG, но не считаем маленькую
разницу доказательством превосходства и пока не меняем значения по умолчанию.

In [4]:
summary = results.groupby("score_coef").agg(
    ndcg_mean=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"),
    ndcg_min=("ndcg@10", "min"), ndcg_max=("ndcg@10", "max"),
    precision_mean=("precision@10", "mean"), recall_mean=("recall@10", "mean"),
    coverage_mean=("coverage@10", "mean"))
display(summary)
candidate = float(summary.ndcg_mean.idxmax())
print("Лучшее среднее NDCG среди проверенных коэффициентов:", candidate)
print("NDCG baseline популярности:", popular_metrics["ndcg@10"])
paired = results.pivot(index="seed", columns="score_coef", values="ndcg@10")
display(paired.subtract(paired[0.2], axis=0).rename_axis(columns="разница с coef=0.2"))
# Парные изменения по пользователям, усреднённые по тем же seed.
delta_rows = []
for coef in [0.0, 0.4]:
    delta = np.mean([per_user[(coef, seed)] - per_user[(0.2, seed)] for seed in SEEDS], axis=0)
    delta_rows.append({"score_coef": coef, "mean_delta": delta.mean(),
                       "лучше": int((delta > 1e-12).sum()),
                       "без изменения": int((np.abs(delta) <= 1e-12).sum()),
                       "хуже": int((delta < -1e-12).sum())})
display(pd.DataFrame(delta_rows).set_index("score_coef"))

,ndcg_mean,ndcg_std,ndcg_min,ndcg_max,precision_mean,recall_mean,coverage_mean
score_coef,,,,,,,
0.000000,0.099762,0.000438,0.099340,0.100216,0.081788,0.064138,"4,242.666667"
0.200000,0.099409,0.000468,0.098891,0.099800,0.081552,0.063972,"4,205.666667"
0.400000,0.099030,0.000584,0.098389,0.099532,0.081375,0.063959,"4,188.000000"


Лучшее среднее NDCG среди проверенных коэффициентов: 0.0
NDCG baseline популярности: 0.11302101779723082


разница с coef=0.2,0.000000,0.200000,0.400000
seed,,,
7,0.000450,0.000000,-0.000502
42,0.000415,0.000000,-0.000268
2026,0.000195,0.000000,-0.000367


,mean_delta,лучше,без изменения,хуже
score_coef,,,,
0.000000,0.000353,1475,4277,1432
0.400000,-0.000379,1261,4588,1335


## Артефакты опыта

Сохраняем таблицы результатов и контекст в `ml/artifacts/als_score_coefficient/`
(вне Git). Полные таблицы также остаются в выводах этого ноутбука. Модель
`als_baseline.npz` и глобальные defaults не перезаписываются.

In [5]:
out = Path.cwd().parent / "artifacts" / "als_score_coefficient"
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / "runs.csv", index=False)
summary.to_csv(out / "summary.csv")
(out / "protocol.json").write_text(json.dumps({
    "snapshot_end": SNAPSHOT_END.isoformat(), "validation_start": VALIDATION_START.isoformat(),
    "test_start": TEST_START.isoformat(), "test_evaluated": False,
    "score_coefficients": COEFFICIENTS, "seeds": SEEDS, "als_params": ALS_PARAMS,
    "min_user_items": 5, "min_item_users": 5,
    "status_weights": DEFAULT_STATUS_WEIGHTS, "min_score": 5, "min_user_scores": 5,
    "clip": [0.5, 1.5], "half_life_days": None,
    "user_id_namespace": "anilist", "item_id_namespace": "anilist",
    "versions": versions, "popular_baseline": popular_metrics,
}, ensure_ascii=False, indent=2) + "\n")
print("Сохранены:", ", ".join(p.name for p in sorted(out.iterdir())))

Сохранены: protocol.json, runs.csv, summary.csv


## Вывод

| score_coef | Средний NDCG@10 | Std между seed |
|---|---:|---:|
| 0 | 0,099762 | 0,000438 |
| 0,2 | 0,099409 | 0,000468 |
| 0,4 | 0,099030 | 0,000584 |

- Коэффициент 0 лучше исходного 0,2 при всех трёх seed. Средняя разница
  +0,000353 (около +0,36%) невелика: сопоставима с разбросом между инициализациями.
  Коэффициент 0,4 немного ухудшает NDCG при каждом из трёх seed.
- Значительного выигрыша от относительной поправки за оценку в этой конфигурации
  не обнаружено. Это **не** означает, что оценки бесполезны: отсечение оценок
  ниже 5 действует во всех вариантах; взаимодействие с другими настройками
  здесь также не исследовалось.
- Популярность за 28 дней имеет NDCG@10 0,113021 — все варианты ALS остаются ниже.
  Одной настройкой score_coef этот разрыв не закрывается.
- Для следующего последовательного опыта принимаем **score_coef=0** как
  предварительную настройку по validation. Менять глобальные defaults и
  перезаписывать исходную модель пока не нужно: настройку передаём явно.
- Следующий опыт — planned 0,3 / 0,5 / 1,0 при score_coef=0, остальных тех же
  параметрах и seed 42 / 7 / 2026. Сравнение остаётся на тех же целях validation.
- Коэффициент 0,2 и seed 42 воспроизвели предыдущий baseline ALS (~0,0998).
  Все девять запусков имеют 3 293 589 положительных пар и полный топ-10
  для 7 184 пользователей. Test не оценивался.
